In [0]:
df = spark.read.table("subscription_pipeline.bronze.fx_rate")
display(df)

In [0]:
bronze_schemapath  = "subscription_pipeline.bronze"
silver_schemapath = "subscription_pipeline.silver"

In [0]:
tables = spark.catalog.listTables(dbName=bronze_schemapath)

In [0]:
fivetran_ingested_column_to_remove = [
    "_file", 	
    "_fivetran_synced", 
    "_modified",
    "_line"
]

In [0]:
def remove_fivetran_ingested_column(df):
    for column in fivetran_ingested_column_to_remove:
        df = df.drop(column)
    return df

In [0]:
# Read from bronze, remove metadata columns, and write to silver
df = spark.read.table(f"{bronze_schemapath}.employee")
df_clean = remove_fivetran_ingested_column(df)
df_clean.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schemapath}.employees")

In [0]:
# Read from bronze, remove metadata columns, and write to silver
df = spark.read.table(f"{bronze_schemapath}.country_master")
df_clean = remove_fivetran_ingested_column(df)
df_clean.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schemapath}.countries")

In [0]:
# Read from bronze, remove metadata columns, and write to silver
df = spark.read.table(f"{bronze_schemapath}.customer")
df_clean = remove_fivetran_ingested_column(df)
df_clean.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schemapath}.customers")

In [0]:
# Read from bronze, remove metadata columns, and write to silver
df = spark.read.table(f"{bronze_schemapath}.opportunity")
df_clean = remove_fivetran_ingested_column(df)
df_clean.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schemapath}.opportunity")

In [0]:
# Read from bronze, remove metadata columns, and write to silver
df = spark.read.table(f"{bronze_schemapath}.product")
df_clean = remove_fivetran_ingested_column(df)
df_clean.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schemapath}.products")

In [0]:
df = spark.read.table(f"{bronze_schemapath}.employee")
display(df)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

def fill_id_column(df, column_name):
    w = Window().orderBy(df.columns[0])
    df = df.withColumn(column_name, row_number().over(w))
    return df

In [0]:
from pyspark.sql.functions import to_date, col, split

df = spark.read.table(f"{silver_schemapath}.employees")
df = fill_id_column(df, "employee_id")
df = df.withColumn("hire_date", to_date(col("hire_date"), "dd-MM-yyyy"))
df = df.withColumn("last_update", to_date(col("last_update"), "dd-MM-yyyy"))
df = df.withColumn(
    "employee_internal_id",
    split(col("employee_name"), " ")[1]
)
df = df.withColumn("is_active_flag", expr("case when is_active_flag = 'Yes' then 1 else 0 end"))

df = df.dropDuplicates()
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schemapath}.employees")

In [0]:
df = spark.read.table(f"{silver_schemapath}.employees")
display(df)

In [0]:
df = spark.read.table(f"{silver_schemapath}.employees")
df_clean = remove_fivetran_ingested_column(df)
df_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schemapath}.employees")

In [0]:
df = spark.read.table(f"{silver_schemapath}.customers")
# df = fill_id_column(df, "customer_id")
df = df.withColumn("account_created_date", to_date(col("account_created_date"), "dd-MM-yyyy"))
df = df.dropDuplicates()
df = df.withColumn(
    "customer_internal_id",
    split(col("customer_name"), " ")[1]
)
display(df)
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schemapath}.customers")

In [0]:
df = spark.read.table(f"{silver_schemapath}.fx_rates")
df = df.withColumn("effective_date", to_date(col("effective_date"), "dd-MM-yyyy"))
df.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schemapath}.fx_rates")

In [0]:
df = spark.read.table(f"{silver_schemapath}.fx_rates")
display(df)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, to_date, col, split, expr

def fill_id_column(df, column_name):
    w = Window().orderBy(df.columns[0])
    df = df.withColumn(column_name, row_number().over(w))
    return df

df = spark.read.table(f"{silver_schemapath}.products")
df = fill_id_column(df, "product_id")
df = df.withColumn("created_date", to_date(col("created_date"), "dd-MM-yyyy"))
df = df.withColumn(
    "product_internal_id",
    split(col("product_name"), " ")[2]
)

df = df.withColumn("is_active", expr("case when is_active = 'Y' then 1 else 0 end"))
df = df.withColumn("list_price", col("list_price").cast("double"))

df = df.dropDuplicates()
display(df)

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schemapath}.products")

In [0]:
from pyspark.sql.functions import to_date, col, row_number, lit, expr, regexp_replace, upper, to_timestamp
from pyspark.sql.window import Window

df = spark.read.table(f"{silver_schemapath}.opportunity")

# Convert date columns to proper date type
df = df.withColumn("start_date", to_date(col("start_date"), "dd-MM-yyyy"))
df = df.withColumn("end_date", to_date(col("end_date"), "dd-MM-yyyy"))

# Remove duplicates
df = df.dropDuplicates()

# Fill missing opportunity_id if necessary, starting with 500000
if "opportunity_id" not in df.columns or df.filter(col("opportunity_id").isNull()).count() > 0:
    w = Window().orderBy(lit(1))
    df = df.withColumn("opportunity_id", row_number().over(w) + 499999)

# Clean revenue_amount: remove nulls and ensure numeric type
df = df.withColumn(
    "revenue_amount",
    regexp_replace(col("revenue_amount"), '[$£]', '')
)
df = df.withColumn(
    "revenue_amount",
    regexp_replace(col("revenue_amount"), '[^0-9\\.-]', '')
)
df = df.withColumn(
    "revenue_amount",
    col("revenue_amount").cast("double")
)
# timestamp upto seconds created_timestamp column
# df = df.withColumn("created_timestamp", try_to_timestamp(col("created_timestamp"), "dd-MM-yyyy HH:mm:ss"))
# Clean close_status: standardize to uppercase
df = df.withColumn("close_status", upper(col("close_status")))

display(df)
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schemapath}.opportunity")

In [0]:
from pyspark.sql.functions import col, countDistinct, isnan, isnull
from pyspark.sql.types import DoubleType, FloatType

silver_tables = [
    "employees",
    "customers",
    "countries",
    "fx_rates",
    "products",
    "opportunity"
]

def cleaning_report(table_name):
    df = spark.read.table(f"{silver_schemapath}.{table_name}")
    total_rows = df.count()
    report = []
    for c in df.columns:
        col_type = df.schema[c].dataType
        if isinstance(col_type, (DoubleType, FloatType)):
            null_count = df.filter(isnull(col(c)) | isnan(col(c))).count()
        else:
            null_count = df.filter(isnull(col(c))).count()
        distinct_count = df.select(countDistinct(col(c))).collect()[0][0]
        report.append({
            "table": table_name,
            "column": c,
            "total_rows": total_rows,
            "null_count": null_count,
            "distinct_count": distinct_count
        })
    return report

all_reports = []
for t in silver_tables:
    all_reports.extend(cleaning_report(t))

report_df = spark.createDataFrame(all_reports)
display(report_df)